# 83 — Campaign C S0 rank-sweep series

次の actionable candidate に対して S0 を実行する。

- 既定 `max_candidates=1`（安全）
- M2 未完了 → `NEED_M2` で停止（73 へ）
- `NO_SELECTION`/`REJECT` → archive して（budget 内なら）次へ
- `SELECTED` → ADVANCE（78 は PR-2 待ち）
- **staged M2 は自動起動しない** / production M6 禁止


In [ ]:
from pathlib import Path
import os, sys, json

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'm7_candidate_queue.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/m7_candidate_queue.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ.setdefault('VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT')
os.environ.setdefault('VALIDATED_RG_M7C_RUN_ID', 'M7-20260720T081500Z-c8d5f02b3c96')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)


In [ ]:
from src.m7_candidate_queue import search_root_for
from src.m7_status import M7_RUN_ID_CAMPAIGN_C
from src.rank_sweep import default_sweep_config
from src.runtime import validate_persistent_root
from src.s0_series import run_s0_series

validate_persistent_root()
M7C_RUN_ID = os.environ.get('VALIDATED_RG_M7C_RUN_ID', M7_RUN_ID_CAMPAIGN_C)
SEARCH_ROOT = search_root_for(PERSIST_ROOT, M7C_RUN_ID)
MAX_CANDIDATES = int(os.environ.get('VALIDATED_RG_S0_SERIES_MAX', '1'))
SWEEP_CONFIG = default_sweep_config(
    max_factor_rank=int(os.environ.get('VALIDATED_RG_SWEEP_MAX_RANK', '96')),
    rank_grid=[16, 24, 25, 32, 36, 48, 49, 64, 81, 96],
    oversampling=16,
    power_iterations=2,
    seed=20260720,
    engineering_margin=0.05,
    sectors_per_shard=8,
)
print('SEARCH_ROOT', SEARCH_ROOT)
print('MAX_CANDIDATES', MAX_CANDIDATES)


In [ ]:
RESULT = run_s0_series(
    project_root=PROJECT_ROOT,
    persistent_root=PERSIST_ROOT,
    search_root=SEARCH_ROOT,
    max_candidates=MAX_CANDIDATES,
    sweep_config=SWEEP_CONFIG,
    materialize_if_missing=True,
)
print(json.dumps({
    k: RESULT.get(k)
    for k in (
        'series_status', 'processed', 'candidate_id', 'selected_rank',
        'next_notebook', 'env_hint', 'note',
    )
    if k in RESULT
}, indent=2, ensure_ascii=False))


In [ ]:
status = RESULT.get('series_status')
if status == 'NEED_M2':
    print('Stop: run notebook 73 with:')
    print(RESULT.get('env_hint'))
elif status == 'SELECTED':
    print('Stop: exploratory SELECTED. Notebook 78 is still PR-2 scaffold.')
    print('Do NOT start production M6.')
elif status in {'ARCHIVED', 'BUDGET_DONE'}:
    print('Budget done / archived. Re-open 82 then 83 for the next candidate.')
elif status == 'EXHAUSTED':
    print('No actionable candidates left. Need new search (notebook 72) or higher j2 stage.')
else:
    print('series_status', status)
